# 02 — Analyse de la qualité des données

## Objectif

Ce notebook évalue la qualité du dataset avant la construction
du pipeline ETL.

Les contrôles portent sur :

- les valeurs manquantes ;
- les doublons ;
- les identifiants ;
- les types ;
- les catégories ;
- les valeurs incohérentes ;
- les colonnes constantes ;
- les colonnes fortement incomplètes.

In [28]:
from pathlib import Path

import polars as pl

In [29]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "source" / "diabetic_data.csv"

In [30]:
df_raw = pl.read_csv(
    DATA_PATH,
    infer_schema_length=10000,
)

In [31]:
df = pl.read_csv(
    DATA_PATH,
    null_values=["?", ""],
    infer_schema_length=10000,
)

In [32]:
null_summary = pl.DataFrame(
    {
        "column": df.columns,
        "null_count": [
            df.select(pl.col(column).null_count()).item()
            for column in df.columns
        ],
    }
).with_columns(
    (
        pl.col("null_count") / df.height * 100
    ).round(2).alias("null_percentage")
).sort("null_percentage", descending=True)

null_summary

column,null_count,null_percentage
str,i64,f64
"""weight""",98569,96.86
"""medical_specialty""",49949,49.08
"""payer_code""",40256,39.56
"""race""",2273,2.23
"""diag_3""",1423,1.4
…,…,…
"""metformin-rosiglitazone""",0,0.0
"""metformin-pioglitazone""",0,0.0
"""change""",0,0.0


In [33]:
null_summary.filter(
    pl.col("null_count") > 0
)

column,null_count,null_percentage
str,i64,f64
"""weight""",98569,96.86
"""medical_specialty""",49949,49.08
"""payer_code""",40256,39.56
"""race""",2273,2.23
"""diag_3""",1423,1.4
"""diag_2""",358,0.35
"""diag_1""",21,0.02


In [34]:
null_summary.with_columns(
    pl.when(pl.col("null_percentage") == 0)
    .then(pl.lit("Aucune valeur manquante"))
    .when(pl.col("null_percentage") < 5)
    .then(pl.lit("Faible"))
    .when(pl.col("null_percentage") < 20)
    .then(pl.lit("Modéré"))
    .when(pl.col("null_percentage") < 50)
    .then(pl.lit("Élevé"))
    .otherwise(pl.lit("Critique"))
    .alias("severity")
)

column,null_count,null_percentage,severity
str,i64,f64,str
"""weight""",98569,96.86,"""Critique"""
"""medical_specialty""",49949,49.08,"""Élevé"""
"""payer_code""",40256,39.56,"""Élevé"""
"""race""",2273,2.23,"""Faible"""
"""diag_3""",1423,1.4,"""Faible"""
…,…,…,…
"""metformin-rosiglitazone""",0,0.0,"""Aucune valeur manquante"""
"""metformin-pioglitazone""",0,0.0,"""Aucune valeur manquante"""
"""change""",0,0.0,"""Aucune valeur manquante"""


In [35]:
question_mark_summary = []

for column in df_raw.columns:
    if df_raw.schema[column] == pl.String:
        count = df_raw.select(
            (pl.col(column) == "?").sum()
        ).item()

        if count > 0:
            question_mark_summary.append(
                {
                    "column": column,
                    "question_mark_count": count,
                    "percentage": round(count / df_raw.height * 100, 2),
                }
            )

question_mark_df = pl.DataFrame(question_mark_summary).sort(
    "percentage",
    descending=True,
)

question_mark_df

column,question_mark_count,percentage
str,i64,f64
"""weight""",98569,96.86
"""medical_specialty""",49949,49.08
"""payer_code""",40256,39.56
"""race""",2273,2.23
"""diag_3""",1423,1.4
"""diag_2""",358,0.35
"""diag_1""",21,0.02


In [36]:
duplicate_row_count = df.height - df.unique().height

print(f"Nombre de lignes dupliquées : {duplicate_row_count}")

Nombre de lignes dupliquées : 0


In [37]:
df_pd = df.to_pandas()

duplicate_row_count = df_pd.duplicated().sum()

print(duplicate_row_count)

0


In [38]:
encounter_duplicates = (
    df.group_by("encounter_id")
    .agg(pl.len().alias("count"))
    .filter(pl.col("count") > 1)
)

encounter_duplicates

encounter_id,count
i64,u32


In [39]:
print(
    f"Nombre d'encounter_id dupliqués : "
    f"{encounter_duplicates.height}"
)

Nombre d'encounter_id dupliqués : 0


In [40]:
patient_duplicates = (
    df.group_by("patient_nbr")
    .agg(pl.len().alias("encounter_count"))
    .filter(pl.col("encounter_count") > 1)
    .sort("encounter_count", descending=True)
)

patient_duplicates.head(20)

patient_nbr,encounter_count
i64,u32
88785891,40
43140906,28
1660293,23
23199021,23
88227540,23
…,…
97391007,19
88479036,19
84348792,18


In [41]:
df.filter(
    (pl.col("time_in_hospital") <= 0)
    | pl.col("time_in_hospital").is_null()
)

encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
i64,i64,str,str,str,str,i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str


In [42]:
df.filter(
    pl.col("num_medications") < 0
)

encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
i64,i64,str,str,str,str,i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str


In [43]:
df.filter(
    pl.col("number_diagnoses") < 0
)

encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
i64,i64,str,str,str,str,i64,i64,i64,i64,str,str,i64,i64,i64,i64,i64,i64,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str


In [44]:
visit_columns = [
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
]

for column in visit_columns:
    invalid_count = df.filter(
        pl.col(column) < 0
    ).height

    print(f"{column} : {invalid_count} valeurs négatives")

number_outpatient : 0 valeurs négatives
number_emergency : 0 valeurs négatives
number_inpatient : 0 valeurs négatives


In [45]:
expected_readmitted_values = {"<30", ">30", "NO"}

actual_readmitted_values = set(
    df.select("readmitted")
    .drop_nulls()
    .to_series()
    .to_list()
)

unexpected_readmitted_values = (
    actual_readmitted_values - expected_readmitted_values
)

print(unexpected_readmitted_values)

set()


In [46]:
df.select(
    pl.col("gender").value_counts(sort=True)
)

gender
struct[2]
"{""Female"",54708}"
"{""Male"",47055}"
"{""Unknown/Invalid"",3}"


In [47]:
expected_medication_values = {
    "No",
    "Steady",
    "Up",
    "Down",
}

In [48]:
medication_columns = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "examide",
    "citoglipton",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
]

existing_medication_columns = [
    column
    for column in medication_columns
    if column in df.columns
]

print("Colonnes de médicaments trouvées :")
print(existing_medication_columns)

Colonnes de médicaments trouvées :
['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']


In [49]:
expected_medication_values = {
    "No",
    "Steady",
    "Up",
    "Down",
}

In [50]:
medication_quality_results = []

for column in existing_medication_columns:
    values = set(
        df.select(column)
        .drop_nulls()
        .to_series()
        .to_list()
    )

    unexpected_values = values - expected_medication_values

    medication_quality_results.append(
        {
            "column": column,
            "unexpected_values": str(unexpected_values),
            "is_valid": len(unexpected_values) == 0,
        }
    )

medication_quality_df = pl.DataFrame(medication_quality_results)

medication_quality_df

column,unexpected_values,is_valid
str,str,bool
"""metformin""","""set()""",true
"""repaglinide""","""set()""",true
"""nateglinide""","""set()""",true
"""chlorpropamide""","""set()""",true
"""glimepiride""","""set()""",true
…,…,…
"""glyburide-metformin""","""set()""",true
"""glipizide-metformin""","""set()""",true
"""glimepiride-pioglitazone""","""set()""",true


In [51]:
high_missing_columns = null_summary.filter(
    pl.col("null_percentage") >= 40
)

high_missing_columns

column,null_count,null_percentage
str,i64,f64
"""weight""",98569,96.86
"""medical_specialty""",49949,49.08


In [52]:
quality_summary = pl.DataFrame(
    {
        "column": df.columns,
        "dtype": [str(df.schema[column]) for column in df.columns],
        "null_count": [
            df.select(pl.col(column).null_count()).item()
            for column in df.columns
        ],
        "unique_count": [
            df.select(pl.col(column).n_unique()).item()
            for column in df.columns
        ],
    }
).with_columns(
    (
        pl.col("null_count") / df.height * 100
    ).round(2).alias("null_percentage")
).sort("null_percentage", descending=True)

quality_summary

column,dtype,null_count,unique_count,null_percentage
str,str,i64,i64,f64
"""weight""","""String""",98569,10,96.86
"""medical_specialty""","""String""",49949,73,49.08
"""payer_code""","""String""",40256,18,39.56
"""race""","""String""",2273,6,2.23
"""diag_3""","""String""",1423,790,1.4
…,…,…,…,…
"""metformin-rosiglitazone""","""String""",0,2,0.0
"""metformin-pioglitazone""","""String""",0,2,0.0
"""change""","""String""",0,2,0.0


In [53]:
REPORT_OUTPUT = (
    PROJECT_ROOT
    / "docs"
    / "data_quality_summary.csv"
)

quality_summary.write_csv(REPORT_OUTPUT)

print(f"Rapport exporté vers : {REPORT_OUTPUT}")

Rapport exporté vers : c:\Users\pc\chu-oujda-rehospitalisation-risk\docs\data_quality_summary.csv


# Conclusion de l’analyse qualité

L’analyse a permis d’identifier les principales anomalies du dataset :

- valeurs manquantes réelles ou représentées par `?` ;
- colonnes fortement incomplètes ;
- variables catégorielles avec valeurs inconnues ;
- codes nécessitant un mapping ;
- diagnostics nécessitant une normalisation ;
- déséquilibre possible de la variable cible ;
- présence de patients avec plusieurs séjours.

Ces observations serviront à concevoir les règles de validation,
la zone de quarantaine et le futur pipeline ETL.